In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from tensorflow.keras import Sequential
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam

# Load Training Dataset

train_data = image_dataset_from_directory(
    "./Intel Image Classification/seg_train",
    image_size=(128, 128),
    batch_size=64
)

# Load Testing Dataset

test_data = image_dataset_from_directory(
    "./Intel Image Classification/seg_test",
    image_size=(128, 128),
    batch_size=64
)

num_classes = len(train_data.class_names)

# Load Pre-trained VGG16 Model

base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(128, 128, 3)
)

# Freeze Base Layers

base_model.trainable = False

# Build Model

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

# Compile Model

model.compile(
    optimizer=Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train Model

history = model.fit(
    train_data,
    epochs=2,
    validation_data=test_data
)

# Compute Loss and Evaluate Model

loss, accuracy = model.evaluate(test_data)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

# Plot Accuracy

plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Training and Validation Accuracy")
plt.legend(["Training", "Validation"])
plt.show()

# Plot Loss

plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Training and Validation Loss")
plt.legend(["Training", "Validation"])
plt.show()

# Visualize Predictions

images, labels = next(iter(test_data))

predictions = model.predict(images)

plt.figure(figsize=(10, 6))

for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(images[i].numpy().astype("uint8"))

    predicted = np.argmax(predictions[i])
    actual = labels[i].numpy()

    plt.title(
        f"Actual: {train_data.class_names[actual]}\n"
        f"Predicted: {train_data.class_names[predicted]}"
    )
    plt.axis("off")

plt.show()

# Fine Tune the Model

base_model.trainable = True

for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_history = model.fit(
    train_data,
    epochs=2,
    validation_data=test_data
)

# Evaluate Fine Tuned Model

loss, accuracy = model.evaluate(test_data)

print("Fine Tuned Test Loss:", loss)
print("Fine Tuned Test Accuracy:", accuracy)
